# Neural network stability：ImageNet / ResNet

在固定 optimizer updates $K$ 下，比较 **SGD、Polyak、Nesterov** 的经验稳定学习率范围如何随 batch size $B$ 和 momentum gap $g=1-\rho$ 变化。

本 notebook 是 [stability_experiment.py](stability_experiment.py) 的交互入口，直接复用模型、优化器、配对种子、稳定性判据和缓存协议。研究对象是 **finite-horizon empirical stability**，不是 uniform stability 或泛化稳定性。完整协议见 [README](README.md)。

默认 `PRESET = "smoke"`：用 32×32 FakeData 跑通训练、汇总与绘图，**不能作为真实 ImageNet 结果**。切换到 `"pilot"` 或 `"full"` 后使用真实数据。所有新结果写入本目录下的 `results/generated/`。

## 1. 环境

使用 Python 3.10+，在仓库根目录安装依赖并启动 Jupyter：

```bash
python -m pip install -e '.[nn,notebooks]'
jupyter lab momentum_nn/stability.ipynb
```

选择安装了 PyTorch / torchvision 的 Python kernel；CUDA 环境先安装与机器匹配的 PyTorch。
从仓库根目录或 `momentum_nn/` 启动均可。notebook 默认 `workers=0`，避免交互环境的数据加载子进程问题。

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

# 支持 kernel 从仓库根目录、notebook 目录或其子目录启动。
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "momentum_nn" / "stability_experiment.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("请从 Momentum_scaling 仓库内启动 notebook。")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".mplconfig"))

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from momentum_nn import stability_experiment as experiment

current_source_hashes = experiment.source_hashes()
if ("IMPORTED_SOURCE_HASHES" in globals()
        and IMPORTED_SOURCE_HASHES != current_source_hashes):
    raise RuntimeError("训练源码已变化，请重启 kernel 并使用新的 RUN_NAME。")
IMPORTED_SOURCE_HASHES = current_source_hashes
print(f"Repository: {REPO_ROOT}")
print(f"Python: {sys.executable}")

## 2. 更新公式与判据

记 $v_k=\theta_k-\theta_{k-1}$，$v_0=0$，每步的随机 batch loss 为 $\ell_k$：

$$
\begin{aligned}
\mathrm{SGD}:\quad &\theta_{k+1}=\theta_k-\eta\nabla\ell_k(\theta_k),\\
\mathrm{Polyak}:\quad &v_{k+1}=\rho v_k-\eta\nabla\ell_k(\theta_k),
\quad \theta_{k+1}=\theta_k+v_{k+1},\\
\mathrm{Nesterov}:\quad &y_k=\theta_k+\rho v_k,\quad
v_{k+1}=\rho v_k-\eta\nabla\ell_k(y_k),
\quad \theta_{k+1}=\theta_k+v_{k+1}.
\end{aligned}
$$

随机初始化 ResNet-18，使用 GroupNorm，全部层训练；恒定绝对学习率、零 weight decay，无 schedule、warmup、梯度裁剪或 AMP。
训练采用有放回抽样及随机增强；同一 seed 配对初始化与抽样种子。
固定训练集 probe 使用中心裁剪，每次在**实际参数位置**测量 cross-entropy；Nesterov 的 train loss 在 look-ahead 位置计算。

记初始化 probe CE 为 $L_0$，最后 `tail_evals` 次均值为 $L_{\rm tail}$，之前同样次数的均值为 $L_{\rm prev}$：

- **survived**：完成 $K$ 次更新且未触发非有限值或计算停止阈值。
- **bounded**：survived，且 $L_{\rm tail}/L_0\le2$、$L_{\rm tail}/L_{\rm prev}\le1.25$。
- **learned**：bounded，且 $L_{\rm tail}/L_0\le0.95$；单独报告。

默认所有 seed 均 bounded 才通过该学习率。计算停止阈值为 probe CE $>100L_0$、gradient norm $>10^6$、parameter norm $>100$ 倍初值；触及有限阈值不等于数学发散。

正式设置要求 $K(1-\rho)\ge30$。固定 $K$ 时样本量 $D=BK$ 随 batch 改变。
两种扫描共享交点，SGD 在 gap 扫描中复用 baseline。这里不预设 NN 谱指数，也不套用线性模型的 $B^\beta$ 拟合。

## 3. 选择实验与调整参数

| PRESET | 数据 / 扫描 | 默认规模 |
|---|---|---|
| `smoke` | FakeData；两个 batch、两个 gap、两个学习率、一个 seed | 16 个去重 trial，每个最多 4 步 |
| `pilot` | ImageNet；$B=4,64$，固定 $g=1/64$，五个学习率、一个 seed | 30 个 trial，每个最多 2000 步 |
| `full` | ImageNet；七个 batch、五个 gap、21 个学习率、三个 seed | 1827 个去重 trial，每个最多 2000 步 |

真实数据目录结构：`DATA_ROOT/train/<class>/*`，仅使用训练集。
`RUN_TRAINING=False` 可先生成清单、查看开销与已有结果；不加载图片或训练模型。
`MAX_NEW_TRIALS=1` 可先执行一个未完成任务，再设为 `None` 并重跑训练单元格继续。

在下面单元格的 `SETTINGS.update(...)` 中改网格、时长和阈值。改变任何协议参数、设备或训练源码时使用新的 `RUN_NAME`；源码改变后重启 kernel。仅改变执行开关不影响缓存。

In [ ]:
PRESET = "smoke"  # "smoke", "pilot", "full"
DATA_ROOT = Path("/path/to/imagenet")
RUN_NAME = f"notebook_{PRESET}_v1"
RUNS_ROOT = REPO_ROOT / "momentum_nn" / "results" / "generated" / RUN_NAME

RUN_TRAINING = True
MAX_NEW_TRIALS = None  # None = 本次运行所有未完成 trial；正整数 = 限制新 trial 数

PRESETS = {
    "smoke": {
        "fake_data": True, "smoke_test": True, "device": "cpu",
        "crop_size": 32, "probe_size": 8, "probe_batch": 4,
        "steps": 4, "eval_every": 1, "tail_evals": 2,
        "batches": "2,4", "fixed_batch": 2, "fixed_gap": 0.25,
        "gaps": "0.25,0.5", "lrs": "0.0001,100", "seeds": "0",
        "max_micro_batch": 1,
    },
    "pilot": {
        "axis": "batch", "batches": "4,64",
        "lrs": "0.001,0.01,0.1,1,8", "seeds": "0",
    },
    "full": {},
}
if PRESET not in PRESETS:
    raise ValueError(f"Unknown preset: {PRESET}")

# 从命令行入口读取同一份默认协议，不读取 Jupyter 的 sys.argv。
args = experiment.parser().parse_args(["sweep", "--runs-root", str(RUNS_ROOT)])
SETTINGS = vars(args)
SETTINGS.update({
    "data_root": "" if PRESET == "smoke" else str(DATA_ROOT.expanduser().resolve()),
    "workers": 0,
})
SETTINGS.update(PRESETS[PRESET])

# 在这里覆盖实验参数；同时为新协议更改 RUN_NAME。
# SETTINGS.update({"axis": "gap", "fixed_batch": 4})
# SETTINGS.update({"steps": 4000, "lrs": "0.01,0.02,0.04,0.08"})

display(pd.DataFrame(
    [(key, str(value)) for key, value in SETTINGS.items()
     if key not in {"command", "runs_root", "shard_count", "shard_index",
                    "dry_run", "no_plot"}],
    columns=["setting", "value"],
))
print(f"Outputs: {RUNS_ROOT}")

## 4. 生成清单并查看开销

此单元格只检查协议、保存清单、读取已完成结果，不加载数据。
`job_index` 是清单中的唯一任务编号；`gap` 是优化器实际使用的 gap，因此 SGD 恒为 1。
同一输出目录必须匹配完整配置和源码 fingerprint，防止混入其他实验缓存。

In [ ]:
if experiment.source_hashes() != IMPORTED_SOURCE_HASHES:
    raise RuntimeError("训练源码已变化，请重启 kernel，并使用新的 RUN_NAME。")
manifest = experiment.make_manifest(args)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = RUNS_ROOT / "manifest.json"
if manifest_path.exists():
    saved = experiment.read_manifest(RUNS_ROOT, verify_source=True)
    if saved["fingerprint"] != manifest["fingerprint"]:
        raise ValueError("输出目录中已有其他协议；请更改 RUN_NAME，再重跑配置和清单。")
else:
    experiment.nn_experiment.atomic_json_dump(manifest, manifest_path)

jobs = pd.DataFrame(manifest["jobs"])
jobs.insert(0, "job_index", range(len(jobs)))
jobs["rho"] = 1.0 - jobs["gap"]
jobs["damping_times"] = manifest["config"]["steps"] * jobs["gap"]
jobs["training_samples_max"] = manifest["config"]["steps"] * jobs["batch"]
jobs["cached"] = [
    experiment.read_result(RUNS_ROOT, manifest, trial) is not None
    for trial in manifest["jobs"]
]

label = "SMOKE TEST — no scientific inference" if manifest["config"]["smoke_test"] else "ImageNet empirical stability"
print(label)
print(f"{len(jobs)} unique trials; {int(jobs['cached'].sum())} cached")
print(f"Up to {len(jobs) * manifest['config']['steps']:,} optimizer updates")
print(f"Up to {int(jobs['training_samples_max'].sum()):,} training samples (+ probe evaluations)")
display(jobs.groupby("algorithm", sort=False).agg(
    trials=("id", "size"),
    cached=("cached", "sum"),
    min_K_gap=("damping_times", "min"),
    training_samples_max=("training_samples_max", "sum"),
))
display(jobs.head(24))

## 5. 执行或续跑

每个 trial 使用一个设备，global batch 按 `max_micro_batch` 做梯度累积。
`auto` 依次选择 CUDA、MPS、CPU；NN 训练统一 float32。

完成的 trial 自动跳过；中断的 trial 下次从同一 seed 重新开始，没有步内 checkpoint。
OOM、数据加载错误等会直接报错，未完成 trial 留作缺失值，不记为算法不稳定。
中断后可直接执行后面的汇总单元格，查看已完成部分；再次执行本单元格继续训练。
不要让两个 kernel 同时运行同一 trial。

In [ ]:
if MAX_NEW_TRIALS is not None and (
    type(MAX_NEW_TRIALS) is not int or MAX_NEW_TRIALS <= 0
):
    raise ValueError("MAX_NEW_TRIALS 必须是正整数或 None。")

if not RUN_TRAINING:
    print("RUN_TRAINING=False：仅查看清单和已有结果。")
else:
    if experiment.source_hashes() != IMPORTED_SOURCE_HASHES:
        raise RuntimeError("训练源码已变化；请重启 kernel 并使用新的 RUN_NAME。")
    current_manifest = experiment.read_manifest(RUNS_ROOT, verify_source=True)
    configured_manifest = experiment.make_manifest(args)
    if not (current_manifest["fingerprint"] == manifest["fingerprint"]
            == configured_manifest["fingerprint"]):
        raise ValueError("配置或清单已变化，请使用新的 RUN_NAME 并重跑配置与清单。")
    config = current_manifest["config"]
    if not config["fake_data"] and not (Path(config["data_root"]) / "train").is_dir():
        raise FileNotFoundError(
            f"找不到 {config['data_root']}/train；请设置 DATA_ROOT，"
            "或用 RUN_TRAINING=False 先查看实验清单。"
        )

    pending = [
        (index, trial) for index, trial in enumerate(current_manifest["jobs"])
        if experiment.read_result(RUNS_ROOT, current_manifest, trial) is None
    ]
    selected = pending if MAX_NEW_TRIALS is None else pending[:MAX_NEW_TRIALS]
    print(f"{len(pending)} pending; executing {len(selected)} new trials")
    started = time.monotonic()
    for number, (index, trial) in enumerate(selected, start=1):
        print(f"\n[{number}/{len(selected)}] job_index={index}", flush=True)
        experiment.run_trial(RUNS_ROOT, current_manifest, trial)
    print(f"Finished this invocation in {time.monotonic() - started:.1f} s")

## 6. 汇总、边界状态与逐 seed 结果

可以在部分任务完成时运行；缺失的学习率或 seed 会产生 `incomplete`。

| status | 含义 |
|---|---|
| `bracketed` | 完整网格先通过后失败，边界夹在 `lower` 与 `upper` 之间 |
| `left_censored` | 全部失败，需向更小学习率扩展 |
| `right_censored` | 全部通过，需向更大学习率扩展 |
| `nonmonotone` | 失败后再次通过，不报告单一边界 |
| `incomplete` | 缺少任意学习率或 seed，暂不报告完整边界 |

边界图中的区间表示学习率网格分辨率，**不是置信区间**；端点间行为仍未知。
`stability_grid` 同时包含 survived / bounded / learned 比例；只有 `complete=True` 的行才纳入完整网格判定。
CSV 中 SGD 的 `gap` 表示扫描坐标，实际 SGD 的 $\rho=0$。

In [ ]:
experiment.summarize(RUNS_ROOT, make_plot=True)
summary = json.loads((RUNS_ROOT / "summary.json").read_text())
boundaries = pd.read_csv(RUNS_ROOT / "boundaries.csv")
per_seed_boundaries = pd.read_csv(RUNS_ROOT / "per_seed_boundaries.csv")
stability_grid = pd.read_csv(RUNS_ROOT / "stability_grid.csv")

print(label)
print(f"Completed {summary['completed']}/{summary['planned']} unique trials")
display(Markdown("**Aggregated boundaries**"))
display(boundaries)
display(Markdown("**Per-seed boundaries**（前 30 行，完整结果在 CSV 中）"))
display(per_seed_boundaries.head(30))
display(Markdown("**Stability fractions**（前 30 行）"))
display(stability_grid.head(30))

## 7. 两种扫描的边界与稳定比例图

直接显示命令行绘图函数保存的 PNG，确保 notebook 与脚本输出一致。
空心三角形表示扫描范围限制，× 表示非单调；不完整点在比例图上显示为灰色。
`pilot` 默认只扫描 batch，因此只生成 batch 图；`smoke` 和 `full` 包含两个方向。
PNG / PDF 同时保存在当前 `RUNS_ROOT`。

In [ ]:
for axis in dict.fromkeys(point["axis"] for point in manifest["points"]):
    display(Markdown(f"### Stability versus {axis}"))
    for stem in (f"boundary_vs_{axis}", f"stable_fraction_vs_{axis}"):
        display(Image(filename=str(RUNS_ROOT / f"{stem}.png"), width=1000))

## 8. 查看逐 trial 结果

下表保留停止原因、实际完成步数和尾部 loss 比例。
`stop_reason` 为空仅表示未触发提前停止，是否 bounded 仍由尾部判据决定。
可以用 `trial_results.query(...)` 挑选算法、batch、gap、学习率和 seed。

In [ ]:
completed_results = {}
rows = []
for index, trial in enumerate(manifest["jobs"]):
    result = experiment.read_result(RUNS_ROOT, manifest, trial)
    if result is None:
        continue
    completed_results[index] = result
    rows.append({
        "job_index": index,
        **trial,
        **{key: result[key] for key in (
            "survived", "bounded", "learned", "updates_completed",
            "tail_loss_ratio", "tail_growth_ratio", "stop_reason", "seconds"
        )},
    })

trial_results = pd.DataFrame(rows)
if trial_results.empty:
    print("尚无已完成 trial；先执行训练单元格。")
else:
    display(trial_results.head(40))
    display(trial_results.assign(
        outcome=trial_results["stop_reason"].replace("", "completed_horizon")
    ).groupby(["algorithm", "outcome"], sort=False).size().rename("trials").to_frame())

## 9. 单次训练轨迹

把 `DIAGNOSTIC_JOB_INDEX` 设为上表的任务编号；默认选第一个完成的 Nesterov trial。
这里只做轨迹诊断，不把单次轨迹当作跨 seed 的稳定边界。

Probe CE、train CE、gradient norm、parameter norm 分开显示：
Nesterov 的 train CE / gradient 位于 look-ahead，probe CE 位于实际参数位置。
probe 图中虚线是**尾部均值判据的参考值**，单个点越线不直接决定 bounded。
日志中的非有限值保留为缺失绘图点，停止原因仍在表格中。

In [ ]:
DIAGNOSTIC_JOB_INDEX = None  # 例如 0；None 优先展示一个已完成的 Nesterov trial

if not completed_results:
    print("尚无已完成 trial 可供绘图。")
else:
    import matplotlib.pyplot as plt

    candidates = [
        index for index, result in completed_results.items()
        if result["trial"]["algorithm"] == "nesterov"
    ]
    diagnostic_index = (
        DIAGNOSTIC_JOB_INDEX if DIAGNOSTIC_JOB_INDEX is not None
        else (candidates[0] if candidates else next(iter(completed_results)))
    )
    if diagnostic_index not in completed_results:
        raise ValueError("该 job_index 尚未完成；请从 trial_results 选择已完成任务。")
    result = completed_results[diagnostic_index]
    trial = result["trial"]
    history_path = experiment.result_path(RUNS_ROOT, trial).with_name("history.jsonl")
    history = pd.DataFrame(
        json.loads(line) for line in history_path.read_text().splitlines() if line.strip()
    )
    display(trial_results.loc[trial_results["job_index"] == diagnostic_index])
    config = manifest["config"]
    location = "lookahead" if trial["algorithm"] == "nesterov" else "base"

    def numeric_series(column):
        values = pd.to_numeric(history[column], errors="coerce")
        return values.replace([np.inf, -np.inf], np.nan)

    with plt.rc_context({"axes.spines.top": False, "axes.spines.right": False}):
        fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
        probe = numeric_series("probe_loss_ratio").dropna()
        axes[0, 0].plot(history.loc[probe.index, "step"], probe, "o-", markersize=3)
        axes[0, 0].axhline(config["stable_loss_ratio"], color="tab:red", ls="--",
                           label="Tail-mean ratio threshold (reference)")
        axes[0, 0].set(ylabel="Probe CE / initial CE (base)")
        axes[0, 0].legend(fontsize=8)
        axes[0, 1].plot(history["step"], numeric_series("train_loss"))
        axes[0, 1].set(ylabel=f"Train CE ({location})")
        axes[1, 0].plot(history["step"], numeric_series("gradient_norm"))
        axes[1, 0].set(ylabel=f"Gradient norm ({location})")
        axes[1, 1].plot(
            history["step"],
            numeric_series("parameter_norm") / result["initial_parameter_norm"],
        )
        axes[1, 1].set(ylabel="Parameter norm / initial norm")
        for ax in axes.flat:
            ax.set_xlabel("Attempted optimizer step")
            ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 4), useOffset=False)
        prefix = "SMOKE TEST | " if config["smoke_test"] else ""
        fig.suptitle(
            f"{prefix}{trial['algorithm']} | B={trial['batch']}, "
            f"gap={trial['gap']:g}, lr={trial['lr']:g}, seed={trial['seed']}\n"
            f"bounded={result['bounded']}, learned={result['learned']}, "
            f"stop={result['stop_reason'] or 'completed_horizon'}"
        )
        diagnostic_path = RUNS_ROOT / f"trajectory_{trial['id']}.png"
        fig.savefig(diagnostic_path, dpi=150)
        plt.close(fig)
    display(Image(filename=str(diagnostic_path), width=1000))

## 10. 保存与正式实验

所有结果已经保存到 `RUNS_ROOT`，无需额外导出：

- `manifest.json`：完整配置、源码 hash、去重任务清单。
- `trials/<id>/config.json`、`history.jsonl`、`summary.json`：数据信息、逐步轨迹及判定。
- `stability_grid.csv`、`boundaries.csv`、`per_seed_boundaries.csv`：比例与边界。
- `boundary_vs_*.png/pdf`、`stable_fraction_vs_*.png/pdf`：两种扫描图。
- `trajectory_<id>.png`：本 notebook 选择的单次诊断图。

真实实验先用 pilot 检查学习率网格是否夹住边界，再固定 full 协议。
边界附近应加密学习率、增加 seed，并用独立目录比较 $K=2000,4000,8000$ 及不同尾部判据。
提前触及停止阈值的 trial 需要重新训练，不能仅靠放宽阈值补成完整运行。

也可以先在 notebook 中设置 `RUN_TRAINING=False` 生成 manifest，
再从仓库根目录把不同 `job-index` 分配给不同 GPU：

```bash
CUDA_VISIBLE_DEVICES=0 python -m momentum_nn.stability_experiment run \
  --runs-root momentum_nn/results/generated/notebook_full_v1 --job-index 0
```

路径与编号应替换成当前清单中的值。每个 trial 只启动一次；全部 worker 结束后重跑汇总及后续单元格。
此 notebook 随附的默认配置仅验证流程，尚未产生真实 ImageNet 的稳定性结论。